# FINAL PROJECT — FROM DATA TO PREDICTION
### Starter Notebook | Supervised Machine Learning

**Participant:** `Lastname Firstname`  
**Selected track:** `Finance / Medicine / Veterinary`  
**Dataset version/source:** `Complete this field`  
**Date:** `Complete this field`

---

## How to use this notebook

This is a **starter notebook**, not a solved practical. It provides the workflow, questions and code scaffolds. You are responsible for implementing each step, interpreting your outputs and justifying your decisions.

1. Read `01_Project_Brief.pdf` before coding.
2. Select exactly one track and use the corresponding dataset.
3. Complete the Markdown answers and code cells marked `TODO`.
4. Keep the test set untouched until the final evaluation.
5. Restart the kernel and run the completed notebook from top to bottom before submission.

**Required models:** DummyClassifier, Logistic Regression, Decision Tree, Random Forest and KNN.  
**Required output:** completed notebook + 12-minute presentation + 3-minute Q&A.

> A model score is not an explanation. Every important modelling decision must be justified in Markdown.

## 0. Project card and track selection

### Choose one track

- **A — Finance:** classify the next-session direction of one selected equity using historical OHLCV data and optional VIX information.
- **B — Medicine:** classify malignant versus benign breast tumours from the Breast Cancer Wisconsin (Diagnostic) dataset.
- **C — Veterinary:** classify disease presence/absence in one ILRI disease workbook (CBPP, FMD or LSD), after checking the target distribution.

### Before coding, answer

1. What real-world question are you investigating?
2. What is the unit of observation (a trading day, a tumour sample, a sampled location/household, ...)?
3. At what point in time is the prediction made?
4. What does the positive class represent?
5. What would count as a serious prediction error in your application?

In [ ]:
# TODO: complete your project metadata
PARTICIPANT = ""
TRACK = ""  # REQUIRED: set to "finance", "medicine", or "veterinary" before loading data
DATASET_SOURCE = ""
TARGET_DESCRIPTION = ""
POSITIVE_CLASS = ""
RANDOM_STATE = 42

## 1. Environment and imports

Use a recent Python environment. Suggested installation (run once if needed):

```bash
pip install numpy pandas matplotlib seaborn scikit-learn openpyxl
```

For finance, download and extract the Zenodo archive manually. For ILRI, download and extract `data_published.zip` from MELDATA. No credentials should be placed in this notebook.

In [ ]:
import sys
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, TimeSeriesSplit, GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## 2. Data acquisition and loading

Select only the block corresponding to your track. Preserve a copy of the raw data and document any file selection.

### Track A — Finance

1. Download the archive from https://zenodo.org/records/20192822.
2. Extract it and inspect the README/manifest and filenames.
3. Select one equity with adequate history. Identify the corresponding stock file and the VIX file (if you use VIX).
4. Parse dates, sort ascending and inspect missing trading days/values. Do not assume that every ticker has the same date coverage.

**Important prediction timing:** prediction is made after the close on day `t`. A feature can use data available through `t`; the target refers to the next trading session. Do not use future values in rolling features.

In [ ]:
# FINANCE — adapt the paths and column names after inspecting the archive.
FINANCE_DATA_DIR = Path("data/finance")  # TODO: change to your extracted folder

# TODO: list available files and inspect the archive README/manifest.
# files = sorted(FINANCE_DATA_DIR.rglob("*.csv"))
# for path in files:
#     print(path.name)

# TODO: load one selected ticker file and, optionally, the VIX file.
# stock = pd.read_csv(FINANCE_DATA_DIR / "YOUR_TICKER.csv")
# stock.head()

### Track B — Medicine / Biology

The dataset is available directly through scikit-learn. If you use the built-in version, verify the mapping in `target_names`. In scikit-learn's Breast Cancer Wisconsin loader, class 0 is malignant and class 1 is benign. For this project, recode malignant as the positive class (`1`). Exclude the sample identifier if you obtain the raw UCI table.

In [ ]:
# MEDICINE — this block loads the dataset from scikit-learn only when TRACK is set to "medicine".
if TRACK.lower() == "medicine":
    bunch = load_breast_cancer(as_frame=True)
    X = bunch.data.copy()
    # In sklearn: target 0 = malignant, target 1 = benign.
    y = (bunch.target == 0).astype(int).rename("malignant")
    print("Target mapping:", {0: "benign (negative)", 1: "malignant (positive)"})
    print("X shape:", X.shape, "| y shape:", y.shape)
    display(X.head())
    display(y.value_counts().rename(index={0:"benign",1:"malignant"}))
else:
    print("Medicine loader skipped. Follow the loader instructions for your selected track.")

### Track C — Veterinary health (ILRI)

Download `data_published.zip` from the official MELDATA page:
https://data.mel.cgiar.org/dataset.xhtml?persistentId=hdl:20.500.11766.1/FK2/8YFO3D

The archive contains `cbpp.xlsx`, `fmd.xlsx` and `lsd.xlsx`. Each workbook also contains a `variable` sheet. First inspect all sheet names and columns. Choose one disease file only after you have counted its positive and negative labels.

The catalogue describes `Weight` as an observation weighting factor. Do not add it to the ordinary predictor matrix by default. Keep a note of its meaning and discuss whether a sample-weighted experiment is justified.

In [ ]:
# VETERINARY — inspect workbook sheets; update the folder path.
ILRI_DATA_DIR = Path("data/veterinary")  # TODO: change to your extracted folder

# TODO: locate the three workbooks and inspect their sheet names.
# for workbook in sorted(ILRI_DATA_DIR.glob("*.xlsx")):
#     print("\nWORKBOOK:", workbook.name)
#     xls = pd.ExcelFile(workbook)
#     print("SHEETS:", xls.sheet_names)

# TODO: select one disease workbook (cbpp.xlsx, fmd.xlsx, or lsd.xlsx).
# disease_file = ILRI_DATA_DIR / "cbpp.xlsx"
# sheets = pd.read_excel(disease_file, sheet_name=None)
# for sheet_name, frame in sheets.items():
#     print(sheet_name, frame.shape, list(frame.columns))

### 2.1 Raw-data audit — complete for your selected track

Before dropping or transforming anything, report:

- number of rows and columns;
- variable names and data types;
- target column and class mapping;
- target counts and proportions;
- missing values per column;
- duplicate rows (and, where relevant, duplicated dates/locations/identifiers);
- data collection period and unit of observation.

**Interpretation prompts:** Is the target imbalanced? Are missing values concentrated in particular variables? Does the dataset contain repeated or grouped observations? What needs to be excluded because it is an ID, post-outcome variable or future information?

In [ ]:
# TODO: assign the correct X and y for your track, then run this audit.
# X = ...  # predictors only; exclude IDs and leakage variables
# y = ...  # binary target; document which class is positive

# assert X is not None and y is not None
# print("Shape:", X.shape)
# print("\nData types:\n", X.dtypes)
# print("\nMissing values:\n", X.isna().sum().sort_values(ascending=False))
# print("\nDuplicates:", X.duplicated().sum())
# print("\nTarget counts:\n", y.value_counts(dropna=False))
# print("\nTarget proportions:\n", y.value_counts(normalize=True, dropna=False))

## 3. Problem formulation

Complete the table in Markdown.

| Element | Your answer |
|---|---|
| Real-world question | TODO |
| Unit of observation | TODO |
| Prediction time | TODO |
| Target variable | TODO |
| Positive class | TODO |
| Candidate predictors | TODO |
| Variables excluded and why | TODO |
| Main error of concern | TODO |
| Primary evaluation metric | TODO |

### Hypothesis before modelling

Which of the four algorithms do you expect to perform best, and why? State a hypothesis based on the geometry, scale, size, noise or structure of your dataset. You must revisit this hypothesis after the experiments.

## 4. Exploratory Data Analysis (EDA)

Do not create plots without explaining what they show. Select plots that answer specific questions.

### Required analyses

1. Target class distribution.
2. Distributions of important numerical variables.
3. Correlation analysis for numerical predictors (where meaningful).
4. At least two comparisons of feature distributions across target classes.
5. A written summary of at least three findings and their implications for preprocessing/model choice.

### Track-specific focus

- **Finance:** price/return history, volatility over time, trading volume, class balance over time. Plot in chronological order.
- **Medicine:** feature scale ranges, strong correlations, distributions for malignant versus benign cases.
- **Veterinary:** class prevalence by disease file, Year_region counts, missingness, environmental distributions and, if appropriate, a geographic plot. Do not interpret associations as causal effects.

In [ ]:
# TODO: implement target distribution and at least three EDA figures.
# Example starting points (adapt to your data):
# y.value_counts().plot(kind="bar", title="Target class counts")
# plt.show()
# X.select_dtypes(include=np.number).hist(figsize=(14, 10), bins=25)
# plt.tight_layout(); plt.show()

### EDA findings — write your interpretation here

1. **Finding:** ...  
   **Evidence:** ...  
   **Decision affected:** ...

2. **Finding:** ...  
   **Evidence:** ...  
   **Decision affected:** ...

3. **Finding:** ...  
   **Evidence:** ...  
   **Decision affected:** ...

## 5. Data preprocessing and feature engineering

Describe the transformations before implementing them.

### Checklist

- Which columns are numerical and which are categorical?
- How will missing numerical values be imputed?
- How will missing categorical values be handled?
- Which categorical columns require encoding?
- Which models require scaling?
- Are any transformations based on the target? If yes, how will leakage be prevented?

**Model-specific principle:** scaling is generally important for KNN and Logistic Regression, because they depend on distances or coefficient optimization affected by feature scales. Decision Trees and Random Forests generally do not require scaling. Imputation and encoding may still be necessary for every model.

In [ ]:
# TODO: define your feature lists after inspecting X.
NUMERIC_FEATURES = []       # e.g. ["feature_1", "feature_2"]
CATEGORICAL_FEATURES = []   # e.g. ["region", "category"]

# TODO: construct two preprocessing pipelines:
# 1) numeric: median imputation; optionally StandardScaler
# 2) categorical: most_frequent imputation + OneHotEncoder(handle_unknown="ignore")
# Use ColumnTransformer to combine them.

## 6. Train / validation / test protocol

The test set is a final estimate of generalization. It is not a tuning set. Split before fitting any data-dependent transformation.

- **Finance:** sort by date; reserve the latest period (for example, final 20%) as test. Use the earlier development period for tuning with `TimeSeriesSplit`. Never shuffle.
- **Medicine:** use `train_test_split(..., stratify=y, random_state=...)`.
- **Veterinary:** use a stratified split for the core experiment. Inspect region/year structure and discuss whether nearby or same region-year observations may make a random split optimistic. A group-based holdout is a valuable extension if the data support it.

Write down the exact split rule and counts. Confirm that both classes are present in each relevant split.

In [ ]:
# TODO: implement a track-appropriate split and define:
# X_train, X_test, y_train, y_test
# For finance, keep date ordering and use an explicit cutoff; do not use train_test_split.
# For medicine/veterinary, use a stratified train_test_split.

# TODO: define the cross-validation object for the development data:
# cv = TimeSeriesSplit(n_splits=5)  # finance
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)  # other tracks

## 7. Baseline model

A baseline provides a reference that a useful model should improve upon. Use `DummyClassifier` with `most_frequent` or `prior`. Do not treat a baseline accuracy as sufficient evidence of model quality, especially with imbalanced classes.

**Tasks:** fit the baseline using the training data, predict the test set, report the required metrics and interpret the result.

In [ ]:
# TODO: fit DummyClassifier using a pipeline if preprocessing is needed.
# baseline = DummyClassifier(strategy="most_frequent")
# baseline.fit(X_train, y_train)
# y_pred_baseline = baseline.predict(X_test)

## 8. Model development

Build a separate Pipeline for each model. Reuse the same train/test observations. Use scaled numerical features for Logistic Regression and KNN; scaling is not generally needed for the tree-based models.

### Required models

1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. K-Nearest Neighbors Classifier

For each one, record: preprocessing, estimator, important hyperparameters, training behaviour and any convergence/warning messages.

In [ ]:
# TODO: create model-specific pipelines using your preprocessing components.
# Example structure:
# logreg_pipe = Pipeline([
#     ("preprocess", preprocessor_scaled),
#     ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
# ])
# tree_pipe = Pipeline([
#     ("preprocess", preprocessor_unscaled),
#     ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
# ])
# rf_pipe = Pipeline([...])
# knn_pipe = Pipeline([...])

## 9. Model selection and hyperparameter tuning

Tune on the development data only. Keep the search space compact and explain why each range was chosen.

Suggested parameters to investigate (adapt them; these are starting points, not mandatory exact grids):

- **Logistic Regression:** `C` and, if justified, `class_weight`.
- **Decision Tree:** `criterion`, `max_depth`, `min_samples_leaf`, `min_samples_split`.
- **Random Forest:** `n_estimators`, `max_features`, `max_depth`, `min_samples_leaf`.
- **KNN:** `n_neighbors`, `weights`, `p` (Minkowski distance).

Choose a scoring metric aligned with your problem. For multiclass extension, use an appropriate macro/weighted scoring rule; the required tracks are binary.

In [ ]:
# TODO: create at least one GridSearchCV per required non-baseline model.
# Use the same CV policy for fair comparison within your track.
# Example pattern:
# search = GridSearchCV(
#     estimator=tree_pipe,
#     param_grid={"model__max_depth": [2, 3, 5, None],
#                 "model__min_samples_leaf": [1, 2, 5]},
#     scoring="f1", cv=cv, n_jobs=-1, refit=True
# )
# search.fit(X_train, y_train)
# print(search.best_params_, search.best_score_)

### Model-selection record

| Model | CV strategy | Scoring metric | Best hyperparameters | Mean CV score |
|---|---|---|---|---|
| Logistic Regression | TODO | TODO | TODO | TODO |
| Decision Tree | TODO | TODO | TODO | TODO |
| Random Forest | TODO | TODO | TODO | TODO |
| KNN | TODO | TODO | TODO | TODO |

Explain why your chosen scoring metric is suitable. Do not compare scores produced by incompatible validation strategies as if they were directly equivalent.

## 10. Final evaluation on the untouched test set

After selecting hyperparameters, evaluate each selected model on the test set exactly once. Report:

- confusion matrix;
- accuracy;
- precision, recall and F1-score for the positive class;
- per-class precision/recall/F1;
- balanced accuracy;
- ROC-AUC and average precision where valid.

For medical data, explicitly name malignant as the positive class. For veterinary data, explicitly name disease presence as the positive class. For finance, explicitly name the positive return direction.

In [ ]:
# TODO: implement a function that returns the metrics required by your track.
# Ensure pos_label is correct for binary precision/recall/F1.
# Do not report ROC-AUC or average precision from hard class labels;
# use predict_proba or decision_function scores.

In [ ]:
# TODO: build a comparison table for Dummy, Logistic Regression,
# Decision Tree, Random Forest and KNN using the same test set.
# Visualize at least one confusion matrix per selected final model.

## 11. Interpretation and error analysis

Answer in Markdown:

1. Which model is best according to your primary metric? Is it also best according to the other metrics?
2. What types of error does the confusion matrix reveal?
3. Does the best model improve meaningfully over the baseline?
4. Which features appear important? Use tree feature importance or permutation importance where appropriate. Remember: predictive importance is not causality.
5. What are the weaknesses of the dataset and evaluation protocol?
6. What evidence would be needed before deploying or using this model in practice?

For finance, distinguish classification performance from economic profitability. For medicine, do not make clinical claims. For veterinary epidemiology, do not infer that environmental associations cause disease.

## 12. Final conclusion

Write a concise conclusion (150–250 words) containing:

- the original prediction question;
- the most important data finding;
- the selected model and reason;
- the principal test metric(s);
- the most important limitation;
- one realistic next step.

## 13. Submission self-check

Before submitting, verify every item:

- [ ] I stated the dataset source and citation.
- [ ] I documented the target and positive class.
- [ ] I removed identifier and leakage variables.
- [ ] I performed and interpreted EDA.
- [ ] My preprocessing is fitted inside pipelines/CV folds.
- [ ] My data split matches the structure of my track.
- [ ] I compared the baseline and all four required algorithms.
- [ ] I tuned hyperparameters without using the test set.
- [ ] I reported a confusion matrix and the required metrics.
- [ ] I interpreted errors and limitations.
- [ ] My notebook runs from top to bottom after path configuration.
- [ ] My 12-minute presentation is ready and I can defend the methodology.